In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from src.data_processor import TennisDataProcessor

In [ ]:
# Load and process data
processor = TennisDataProcessor()
matches = processor.load_matches_data(data_path="../data/raw/")
clean_matches = processor.clean_matches_data()

In [ ]:
# Calculate H2H records

print("=== HEAD-TO-HEAD CALCULATION TEST ===")

h2h_records = processor.calculate_head_to_head()

print(f"H2H records generated: {len(h2h_records)} player pairs")
print(f"Total matches in dataset: {len(processor.matches_df)}")

# Verify total match count consistency
total_h2h_matches = sum(record['total_matches'] for record in h2h_records.values())
print(f"Sum of H2H matches: {total_h2h_matches}")

if total_h2h_matches == len(processor.matches_df):
    print("PASS: Match count consistency verified")
else:
    print("FAIL: Match counts do not match")

In [ ]:
# Test player pairing
print("=== PLAYER PAIRING CONSISTENCY ===")

sample_pairs = list(h2h_records.keys())[:5]
inconsistent_pairs = []

print("Sample player pairs:")
for pair in sample_pairs:
    player1, player2 = pair
    is_sorted = player1 < player2
    status = "PASS" if is_sorted else "FAIL"
    print(f"{status}: ({player1}, {player2})")
    
    if not is_sorted:
        inconsistent_pairs.append(pair)

# Test win/loss arithmetic
print(f"\n=== WIN COUNT VALIDATION ===")
validation_errors = 0

for pair, record in list(h2h_records.items())[:3]:
    total_wins = record['player1_wins'] + record['player2_wins']
    total_matches = record['total_matches']

    if total_wins == total_matches:
        print(f"PASS: {pair[0]} vs {pair[1]} - {total_wins} wins = {total_matches} matches")
    else:
        print(f"FAIL: {pair[0]} vs {pair[1]} - wins({total_wins}) != matches({total_matches})")
        validation_errors += 1

if validation_errors == 0:
    print("All win counts consistent with match totals")

In [ ]:
print("=== SURFACE-SPECIFIC VALIDATION ===")

# Find multi-surface rivalries
multi_surface_pairs = [(pair, record) for pair, record in h2h_records.items() 
                       if len(record['surfaces']) > 1]

if multi_surface_pairs:
    pair, record = multi_surface_pairs[0]
    player1, player2 = pair
    
    print(f"Testing: {player1} vs {player2}")
    print(f"Total matches: {record['total_matches']}")
    print(f"Surfaces: {list(record['surfaces'].keys())}")
    
    # Validate surface totals
    surface_total = 0
    for surface, surface_record in record['surfaces'].items():
        matches_on_surface = surface_record['player1_wins'] + surface_record['player2_wins']
        surface_total += matches_on_surface
        print(f"  {surface}: {matches_on_surface} matches")
    
    if surface_total == record['total_matches']:
        print("PASS: Surface totals match overall total")
    else:
        print(f"FAIL: Surface total ({surface_total}) != overall total ({record['total_matches']})")
else:
    print("No multi-surface rivalries found")

In [ ]:
print("=== TEMPORAL VALIDATION ===")

# Verify last_match_date accuracy
test_pairs = list(h2h_records.items())[:3]

for pair, record in test_pairs:
    player1, player2 = pair
    recorded_last_date = record['last_match_date']
    
    # Find actual most recent match
    pair_matches = processor.matches_df[
        ((processor.matches_df['winner_name'] == player1) & (processor.matches_df['loser_name'] == player2)) |
        ((processor.matches_df['winner_name'] == player2) & (processor.matches_df['loser_name'] == player1))
    ]
    
    if len(pair_matches) > 0:
        actual_last_date = pair_matches['tourney_date'].max()
        
        if recorded_last_date == actual_last_date:
            print(f"PASS: {player1} vs {player2} - dates match")
        else:
            print(f"FAIL: {player1} vs {player2}")
            print(f"  Recorded: {recorded_last_date}")
            print(f"  Actual: {actual_last_date}")

In [ ]:
print("=== STATISTICAL SUMMARY ===")

match_counts = [record['total_matches'] for record in h2h_records.values()]

print(f"Total player pairs: {len(h2h_records)}")
print(f"Mean matches per pair: {np.mean(match_counts):.1f}")
print(f"Max matches (biggest rivalry): {max(match_counts)}")

# Top rivalries
biggest_rivalries = sorted(h2h_records.items(), key=lambda x: x[1]['total_matches'], reverse=True)[:3]

print(f"\nTop rivalries:")
for pair, record in biggest_rivalries:
    player1, player2 = pair
    print(f"  {player1} vs {player2}: {record['total_matches']} matches")

# Edge cases
single_match_pairs = sum(1 for record in h2h_records.values() if record['total_matches'] == 1)
print(f"\nPairs with single match: {single_match_pairs}")